# CS182 decoder baselines

Five predictors for one target: Index-finger velocity (`Index_vx_cm`, `Index_vy_cm`) in 50 ms bins.

| Section | Predictor | What it tests |
|---|---|---|
| 5 | Ridge | Linear baseline; separates artifact cleaning from label-based adaptation |
| 6 | KNN | Retrieval from the k labeled ON trials, with no weight updates |
| 7 | MLP | Nonlinear capacity without temporal context |
| 8 | GRU | Temporal sequence model over the 40-bin trial |
| 9 | EEGNet | Learned temporal and spatial filters from raw 1 kHz LFP |

Sections 0-4 are shared: data, metrics, training utilities, and evaluation utilities. Each predictor section has its own model, training, and evaluation cells.

**Protocol.** Train on 14 stimulation-off sessions. Validate on 3 stimulation-off validation sessions. Test on 14 stimulation-on held-out sessions. For each test session, `k` labeled ON trials come from the calibration pool (blocks 1-2); the fixed 30-trial query set comes from blocks 3-5. The same-day no-stim session is the OFF ceiling.

**Training data.** Every predictor trains on the 14 stimulation-off sessions. Only the ridge refit and KNN k-shot conditions use labeled ON trials, and they use them at evaluation time. Fine-tuning, LoRA, and the in-context transformer are not implemented in this notebook.

**What the baselines should show.** At 20 Hz, the no-adaptation decoder should fall below zero; refitting on k=20 ON trials should recover most of the loss. PARRM alone should not. At 3 Hz, the loss should be small. KNN should approach ridge at k=20 without weight updates. The OFF reference sets the ceiling, and per-bin velocity keeps every R2 small.


## 0. Setup

### Imports

In [ ]:
import glob
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.neighbors import KNeighborsRegressor


## 1. Data and splits

Features are 35 channels x 8 features per 50 ms bin. `export_none` is raw LFP; `parrm_none` is PARRM-cleaned. Features are median/MAD normalized using training trials or the calibration pool, never the query trials.

**Shapes.** `T` is the number of trials in the split: 40 train, 10 validation, 30 query, 50 full session. Each trial is 40 bins of 50 ms, covering -0.5 s to +1.5 s around the go cue.

| Key | Shape | Meaning |
|---|---|---|
| `features` (`lag_count=0`) | `(T, 40, 280)` | 35 channels x 8 features per bin |
| `features` (`lag_count=9`) | `(T, 40, 2800)` | 10 bins of history x 280 |
| `targets` | `(T, 40, 2)` | `Index_vx_cm`, `Index_vy_cm` per bin |
| `valid_bins` | `(T, 40)` | true where features and targets are finite |
| `bin_ids` | `(T, 40)` | global 50 ms bin index |
| `trials` | `(T,)` | trial numbers |

`valid_windows` flattens the first two axes to `(N_valid, 2800)` and `(N_valid, 2)`. The raw-LFP loader returns `(T, 35, 2000)` instead: 35 channels x 2000 samples (2 s at 1 kHz).

`features` is overloaded: for ridge, KNN, and MLP it is the 280-dim band-power tensor; for EEGNet it is raw LFP. `targets` is always velocity.



In [ ]:
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
DRAWS_PATH = os.path.join(RESULTS_DIR, 'draws.json')
BIN_S = 0.05  # 50 ms bins

sys.path.insert(0, PROJECT_ROOT)

import EECS182_NHP_data_loader as lab_loader
import EECS182_NHP_features as lab_features

lab_paths = lab_loader.Paths(
    root=DATA_DIR,
    export_dir=os.path.join(DATA_DIR, 'exports', 'export_h5_LFPs'),
    trial_dir=os.path.join(DATA_DIR, 'exports', 'export_trials'),
    parrm_dir=os.path.join(DATA_DIR, 'parrm', 'parrm_h5_LFPs'),
    kin_dir=os.path.join(DATA_DIR, 'kinematics_bins50ms', 'bins50ms'),
)

MANIFEST_PATH = os.path.join(PROJECT_ROOT, 'EECS182_data_manifest.csv')
SPLITS_PATH = os.path.join(PROJECT_ROOT, 'splits.json')
MANIFEST_TABLE = lab_loader.load_manifest(lab_paths, MANIFEST_PATH)
SESSION_TABLE = MANIFEST_TABLE.set_index('session')  # metadata, sessions stored row-wise
SPLITS = lab_loader.load_splits(lab_paths, SPLITS_PATH)  # fixed train/val/test and calibration splits


def read_json(path):
    """Read a JSON file and return its parsed contents."""
    with open(path) as file:
        return json.load(file)


def load_kinematics(session):
    """Load one session's 50 ms kinematics with screen-cm calibration."""
    return lab_loader.load_kinematics(session, lab_paths)


def load_session_metadata(session):
    """Load one session's manifest row and trial table without the LFP."""
    return lab_loader.load_session(session, lab_paths, with_lfp=False, manifest=MANIFEST_TABLE)


def load_split_trials(session, part):
    """Return the included trial numbers for one train, validation, or calibration part."""
    return lab_loader.split_trials(SPLITS, load_session_metadata(session), part)


Y_COLS = ('Index_vx_cm', 'Index_vy_cm')  # target velocities
PAD_S = 1.0  # extra seconds for normalization
MAD_TO_STD = 1.4826  # MAD to standard deviation


def load_raw_features(session, variant):
    """Load one session's cached features without normalization."""
    return lab_features.load_features(session, lab_paths, variant=variant,
                                      normalize=False)


def _select_trial_rows(session_metadata, trials):
    """Return the manifest trial rows for the requested trial numbers, sorted by go cue."""
    if trials is None:
        return session_metadata.included
    trial_numbers = {int(trial) for trial in trials}
    selected = session_metadata.trials[session_metadata.trials.trial.isin(trial_numbers)]
    return selected.sort_values('t_go')


def bins_within_trial_span(trial_rows, bin_ids):
    """Mask the bins that fall inside the included trials plus the normalization padding."""
    start = float(trial_rows.t_go.min() - PAD_S)
    stop = float(trial_rows.t_reward.max() + PAD_S)
    return (bin_ids * BIN_S >= start) & ((bin_ids + 1) * BIN_S <= stop)


def normalize_features(session, variant, normalization_trials):
    """Normalize each feature by its median and MAD over the requested trials."""
    raw = load_raw_features(session, variant)
    trial_rows = _select_trial_rows(load_session_metadata(session), normalization_trials)
    reference = raw['X'][bins_within_trial_span(trial_rows, raw['bin_id'])]
    median = np.nanmedian(reference, axis=0)
    scale = MAD_TO_STD * np.nanmedian(np.abs(reference - median), axis=0)
    scale = np.where(scale > 0, scale, 1.0)
    return raw | {'X': ((raw['X'] - median) / scale).astype(np.float32)}


def load_trial_dataset(session, trials, variant, normalization_trials, lag_count):
    """Cut features and kinematics into aligned trial arrays for one session."""
    features = normalize_features(session, variant, normalization_trials)
    trial_rows = _select_trial_rows(load_session_metadata(session), trials)
    inputs, targets, valid_bins, bin_ids = lab_features.trial_arrays(
        features, trial_rows, kin=load_kinematics(session), align='t_go', pre=0.5, post=1.5,
        n_lags=lag_count, kin_cols=Y_COLS)
    return dict(features=inputs, targets=targets, valid_bins=valid_bins, bin_ids=bin_ids,
                trials=trial_rows.trial.to_numpy())


def load_split_dataset(session, part, variant, lag_count):
    """Load one session's (T, 40, F) features and (T, 40, 2) y for a split part."""
    trials = load_split_trials(session, part)
    normalization_trials = (trials if part in ('train', 'val')
                            else SPLITS['calibration'][session]['pool'])
    return load_trial_dataset(session, trials, variant, normalization_trials, lag_count)


def load_off_reference_dataset(stimulated_session, variant, lag_count):
    """Load the full no-stimulation reference session for one stimulated session."""
    off_session = SPLITS['off_reference'][stimulated_session]
    trials = load_session_metadata(off_session).included.trial.astype(int).tolist()
    return load_trial_dataset(off_session, trials, variant, trials, lag_count)


def valid_windows(dataset):
    """Flatten the valid bins into (N, features) and (N, targets)."""
    return dataset['features'][dataset['valid_bins']], dataset['targets'][dataset['valid_bins']]


def load_lfp_dataset(session, trials):
    """Load raw 1 kHz LFP as (T, 35, 2000) with velocity targets."""
    session_data = lab_loader.load_session(session, lab_paths, with_lfp=True, manifest=MANIFEST_TABLE)
    lab_loader.normalize(session_data)
    session_data.trials = session_data.trials[session_data.trials.include == 1]
    examples = lab_loader.trial_examples(
        session_data, load_kinematics(session), align='t_go', pre=0.5, post=1.5,
        trials=trials, kin_cols=Y_COLS)
    lfp = [example['lfp'].reshape(example['lfp'].shape[0], -1) for example in examples]
    return dict(features=np.stack(lfp).astype(np.float32),
                targets=np.stack([example['kin'] for example in examples]).astype(np.float32),
                valid_bins=np.stack([example['kin_valid'] for example in examples]).astype(bool))


def split_sessions(part, stimulated):
    """Return the sessions in one split part whose stimulation flag matches."""
    return [s for s in SPLITS['sessions'][part] if int(SESSION_TABLE.loc[s, 'stim']) == stimulated]


# Stimulation-off sessions used to fit the decoders.
OFF_TRAIN_SESSIONS = split_sessions('train', 0)
K_SHOTS = (2, 5, 10, 20)  # calibration trial counts


def _as_ints(trials):
    """Convert trial numbers to Python integers."""
    return [int(trial) for trial in trials]


def build_draws():
    """Build the nested k-shot calibration draws for every stimulated test session."""
    sessions = {}
    for session in split_sessions('test', 1):
        calibration = SPLITS['calibration'][session]
        sessions[session] = dict(
            pool=_as_ints(calibration['pool']),
            query=_as_ints(calibration['query']),
            subsets={str(size): [_as_ints(draw) for draw in calibration['subsets'][str(size)]]
                     for size in K_SHOTS},
        )
    return dict(ks=list(K_SHOTS), sessions=sessions)


def run_draws(out=DRAWS_PATH):
    """Write the nested calibration draws to JSON."""
    with open(out, 'w') as file:
        json.dump(build_draws(), file, indent=1)


## 2. Metrics

`R²` is computed separately for `vx` and `vy`, then averaged. `MIN_R2_GAP` ignores gaps below `0.001` when computing recovery.


In [ ]:
MIN_R2_GAP = 1e-3  # ignore R² gaps smaller than this


def r2_per_output(y, y_hat, valid=None):
    """Return R² for vx and vy separately, with NaN where undefined."""
    y = np.asarray(y, np.float64)
    y_hat = np.asarray(y_hat, np.float64)
    if valid is not None:
        valid = np.asarray(valid, bool)
        y, y_hat = y[valid], y_hat[valid]
    if len(y) == 0:
        return np.full(y.shape[1], np.nan)
    with np.errstate(invalid='ignore', divide='ignore'):
        scores = r2_score(y, y_hat, multioutput='raw_values', force_finite=False)
    return np.where(np.isfinite(scores), scores, np.nan)


def mean_r2(y, y_hat, valid=None):
    """Average R² over the two outputs, vx and vy."""
    return float(np.nanmean(r2_per_output(y, y_hat, valid)))


def recovery(r2_model, r2_base, r2_off):
    """Fraction of the stimulation-induced R² gap recovered."""
    r2_model = np.asarray(r2_model, np.float64)
    r2_base = np.asarray(r2_base, np.float64)
    gap = np.asarray(r2_off, np.float64) - r2_base
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(np.abs(gap) > MIN_R2_GAP, (r2_model - r2_base) / gap, np.nan)


## 3. Shared training utilities

In [ ]:
LR = 1e-3
CLIP_NORM = 1.0
MIN_IMPROVEMENT = 1e-4
TRIAL_BATCH_SIZE = 16
WINDOW_BATCH_SIZE = 256
CHECKPOINT_DIR = os.path.join(RESULTS_DIR, 'checkpoints')


def save_checkpoint(model, name):
    """Save a PyTorch model state dict."""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f'{name}.pt'))


def save_ridge(decoder, name):
    """Save ridge weights, intercept, and alpha."""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    np.savez(os.path.join(CHECKPOINT_DIR, f'{name}.npz'),
             coefficients=decoder.coefficients, intercept=decoder.intercept, alpha=decoder.alpha)


def select_device():
    """Return the best available PyTorch device."""
    if torch.backends.mps.is_available():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def masked_mse(y_hat, y, valid):
    """MSE over valid bins, averaged over vx and vy."""
    if not valid.any():
        return y_hat.sum() * 0.0
    return F.mse_loss(y_hat[valid], y[valid])


def _to_tensors(arrays):
    """Convert numpy arrays to float or bool tensors."""
    return [torch.as_tensor(np.asarray(array),
                            dtype=torch.bool if np.asarray(array).dtype == bool else torch.float32)
            for array in arrays]


def predict(model, X):
    """Run one forward pass over the whole array."""
    device = select_device()
    model = model.to(device).eval()
    X = torch.as_tensor(np.asarray(X), dtype=torch.float32).to(device)
    with torch.inference_mode():
        return model(X).cpu().numpy()


def _train_epoch(model, tensors, loss_fn, optimizer, order, batch_size, device):
    """Run one epoch and return the sample-weighted mean loss."""
    model.train()
    loss_sum = 0.0
    for start in range(0, len(order), batch_size):
        batch = order[start:start + batch_size]
        X, *y = [tensor[batch].to(device) for tensor in tensors]
        loss = loss_fn(model(X), *y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        optimizer.step()
        loss_sum += float(loss.detach()) * len(batch)
    return loss_sum / len(order)


def _fit(model, train, val, loss_fn, seed, epochs, weight_decay, batch_size, patience):
    """Fit one model, stop on validation R², and restore the best weights."""

    # Set device, seeds, tensors, and the AdamW optimizer.
    device = select_device()
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = model.to(device)
    tensors = _to_tensors(train)
    X_val, *y_val = [tensor.numpy() for tensor in _to_tensors(val)]
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=weight_decay)

    # Early-stopping training loop.
    best_r2, best_epoch, best_weights, bad_epochs = -np.inf, 0, None, 0
    training_log = []
    for epoch in range(1, epochs + 1):
        order = rng.permutation(len(tensors[0]))  # shuffle each epoch to avoid trial-order bias
        train_loss = _train_epoch(model, tensors, loss_fn, optimizer, order, batch_size, device)
        y_hat = predict(model, X_val)
        val_r2 = mean_r2(y_val[0], y_hat, *y_val[1:])
        training_log.append(dict(epoch=epoch, train_loss=train_loss, val_r2=val_r2))

        if val_r2 > best_r2 + MIN_IMPROVEMENT:
            best_r2, best_epoch, bad_epochs = val_r2, epoch, 0
            best_weights = {
                name: value.detach().clone()
                for name, value in model.state_dict().items()
            }
            continue

        bad_epochs += 1
        if bad_epochs >= patience:
            break

    if best_weights is not None:
        model.load_state_dict(best_weights)
    model.eval()
    return model, dict(training_log=training_log, best_val_r2=best_r2, best_epoch=best_epoch)


def train_trial(model, train, val, seed, epochs=150):
    """Train a model on (trials, bins, features) arrays."""
    keys = ('features', 'targets', 'valid_bins')
    return _fit(
        model,
        [train[key] for key in keys],
        [val[key] for key in keys],
        masked_mse, seed, epochs, weight_decay=1e-4, batch_size=TRIAL_BATCH_SIZE, patience=20)


def train_window(model, train, val, seed):
    """Train a model on flat (windows, features) arrays."""
    return _fit(
        model, train, val, F.mse_loss, seed, epochs=200, weight_decay=1e-2,
        batch_size=WINDOW_BATCH_SIZE, patience=15)


def predict_trials(model, data):
    """Predict all trial bins as (trials, bins, 2)."""
    return predict(model, data['features'])




## 4. Shared evaluation utilities

These helpers cut windows, build the k-shot draws, and score a decoder on the query and OFF reference. They do not belong to any one predictor.

In [ ]:
RAW_VARIANT = 'export_none'
PARRM_VARIANT = 'parrm_none'
VARIANTS = ((RAW_VARIANT, 'raw'), (PARRM_VARIANT, 'parrm'))


def collect_windows(sessions, part, variant, lags):
    """Concatenate valid (X, y) windows from sessions."""
    pairs = [valid_windows(load_split_dataset(s, part, variant, lags)) for s in sessions]
    X, y = zip(*pairs)
    return np.concatenate(X), np.concatenate(y)


def load_query_and_off_windows(session, variant, lags):
    """Return query and same-day OFF (X, y) windows."""
    query = load_split_dataset(session, 'query', variant, lags)
    off = load_off_reference_dataset(session, variant, lags)
    return valid_windows(query), valid_windows(off)


def load_calibration_windows(session, trials, pool, variant, lags):
    """Return (X, y) for one calibration draw, with features normalized on the pool."""
    return valid_windows(load_trial_dataset(session, trials, variant, pool, lags))


def calibration_draws(draws, session, variant, lags):
    """Yield (k, draw, (X, y)) for every k-shot draw of one session."""
    entry = draws['sessions'][session]
    for k in draws['ks']:
        for draw, trials in enumerate(entry['subsets'][str(k)]):
            yield k, draw, load_calibration_windows(session, trials, entry['pool'], variant, lags)


def score_row(session, method, feature_tag, y, y_hat, y_off, y_hat_off, k=0, draw=0, seed=0):
    """Build one metrics row from query and OFF predictions."""
    r2 = r2_per_output(y, y_hat)
    r2_off = r2_per_output(y_off, y_hat_off)
    return dict(method=method, feature_variant=feature_tag, session=session,
                am_freq=float(SESSION_TABLE.loc[session, 'am_freq_hz']), k=k, draw=draw, seed=seed,
                r2_vx=float(r2[0]), r2_vy=float(r2[1]),
                r2_ave=float(np.nanmean(r2)),
                r2_off_vx=float(r2_off[0]), r2_off_vy=float(r2_off[1]),
                r2_off_ave=float(np.nanmean(r2_off)),
                n_bins=len(y))


def score_decoder(decoder, session, method, feature_tag, query, off, k=0, draw=0):
    """Score one fitted decoder on query and OFF windows."""
    X, y = query
    X_off, y_off = off
    return score_row(session, method, feature_tag, y, decoder.predict(X),
                     y_off, decoder.predict(X_off), k, draw)


def save_training_log(summary, method, seed, training_log_dir):
    """Write one per-seed training log as JSON."""
    if not training_log_dir:
        return
    os.makedirs(training_log_dir, exist_ok=True)
    with open(os.path.join(training_log_dir, f'{method}_seed{seed}.json'), 'w') as file:
        json.dump(dict(method=method, seed=seed, **summary), file)


def concat_off_train(load_part, part, excluded_session=None):
    """Concatenate trial arrays from the stimulation-off training sessions."""
    datasets = [load_part(s, part) for s in OFF_TRAIN_SESSIONS if s != excluded_session]
    return {key: np.concatenate([d[key] for d in datasets])
            for key in ('features', 'targets', 'valid_bins')}


def score_trial(model, session, method, feature_tag, seed, query, off):
    """Score one trial model on query and OFF datasets."""
    y_hat = predict_trials(model, query)[query['valid_bins']]
    y_hat_off = predict_trials(model, off)[off['valid_bins']]
    return score_row(session, method, feature_tag,
                     query['targets'][query['valid_bins']], y_hat,
                     off['targets'][off['valid_bins']], y_hat_off, seed=seed)


## 5. Ridge Regression

Ridge fits a linear map $W$ and intercept $b$ by minimizing

$$
\min_W \ \|Y - XW\|_F^2 + \alpha \|W\|_F^2
$$

Solve directly for $W$:

$$
(X^\top X + \alpha I)W = X^\top Y
$$


In [ ]:
class RidgeRegressor:
    def __init__(self, alpha):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, np.float64)
        y = np.asarray(y, np.float64)

        N_features = X.shape[1]

        x_mean = X.mean(axis=0)
        y_mean = y.mean(axis=0)
        X_centred = X - x_mean
        y_centred = y - y_mean

        # Ridge normal equations: (X^T X + alpha I) W = X^T Y.
        system = X_centred.T @ X_centred + self.alpha * np.eye(N_features)
        self.coefficients = np.linalg.solve(system, X_centred.T @ y_centred)
        self.intercept = y_mean - x_mean @ self.coefficients
        return self

    def predict(self, X):
        return np.asarray(X, np.float64) @ self.coefficients + self.intercept


### 5.1 Training

`ALPHAS` sweep; the penalty is chosen by mean validation R² across the 6 stimulation-on validation sessions.


In [ ]:
ALPHAS = (100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000)  # ridge penalties


def tune_ridge_alpha(X_train, y_train, variant, lag_count):
    """Pick the penalty with the best mean validation R² over the ON validation sessions."""
    validation = [valid_windows(load_split_dataset(s, 'query', variant, lag_count))
                  for s in split_sessions('val', 1)]

    def score(alpha):
        """Average over sessions; each session is already averaged over vx and vy."""
        decoder = RidgeRegressor(alpha).fit(X_train, y_train)
        return np.nanmean([mean_r2(y, decoder.predict(X)) for X, y in validation])

    return max(ALPHAS, key=score)


def train_ridge(variant, lag_count):
    X_train, y_train = collect_windows(OFF_TRAIN_SESSIONS, 'train', variant, lag_count)
    alpha = tune_ridge_alpha(X_train, y_train, variant, lag_count)
    return RidgeRegressor(alpha).fit(X_train, y_train), alpha


### 5.2 Evaluation

Score the frozen decoder on each ON query set, and refit a copy on the k calibration trials for every draw.


In [ ]:
def evaluate_ridge(draws, decoder, alpha, variant, feature_tag, lag_count, sessions):
    rows = []
    for session in sessions:
        query, off = load_query_and_off_windows(session, variant, lag_count)
        for k, draw, cal in calibration_draws(draws, session, variant, lag_count):
            refit = RidgeRegressor(alpha).fit(*cal)
            rows.append(score_decoder(refit, session, f'ridge_{feature_tag}_refit',
                                      feature_tag, query, off, k, draw))
        rows.append(score_decoder(decoder, session, f'ridge_{feature_tag}_off',
                                  feature_tag, query, off))
    return rows


def run_ridge(draws, variant, feature_tag, lag_count, sessions):
    decoder, alpha = train_ridge(variant, lag_count)
    save_ridge(decoder, f'ridge_{feature_tag}')
    return evaluate_ridge(draws, decoder, alpha, variant, feature_tag, lag_count, sessions)


### 5.3 Sanity check: one session, 280 inputs, 2 outputs

The smallest end-to-end example. It uses one stimulation-off training session, one block of held-out trials, and no lags. R² near zero is expected for per-bin velocity; the full baseline uses 9 lags and cross-session training.


In [ ]:
SANITY_SESSION = OFF_TRAIN_SESSIONS[0]
SANITY_VARIANT = 'export_none'

sanity_train = load_split_dataset(SANITY_SESSION, 'train', SANITY_VARIANT, 0)
sanity_val = load_split_dataset(SANITY_SESSION, 'val', SANITY_VARIANT, 0)
X_train, y_train = valid_windows(sanity_train)
X_val, y_val = valid_windows(sanity_val)

print('session       ', SANITY_SESSION)
print('X_train       ', X_train.shape, ' y_train', y_train.shape)
print('X_val         ', X_val.shape, ' y_val  ', y_val.shape)
print('targets       ', Y_COLS)

sanity_model = RidgeRegressor(1e3).fit(X_train, y_train)
sanity_pred = sanity_model.predict(X_val)
print('R2 (vx, vy)   ', r2_per_output(y_val, sanity_pred))
print('R2 mean       ', mean_r2(y_val, sanity_pred))
print('one bin       ')
print('  x[0][:8]    ', X_val[0, :8])
print('  y[0]        ', y_val[0])
print('  y_hat[0]     ', sanity_pred[0])

## 6. KNN

In [ ]:
N_PCA = 48  # PCA dimensions before KNN
NEIGHBORS = (5, 10, 25, 50, 100, 200)  # KNN neighbours to try


class KNNRegressor:
    """PCA followed by inverse-distance KNN regression."""

    def __init__(self, n_neighbors):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.pca = PCA(n_components=N_PCA, random_state=0).fit(X)
        self.model = KNeighborsRegressor(
            n_neighbors=min(self.n_neighbors, len(X)), weights='distance'
        ).fit(self.pca.transform(X), y)
        return self

    def predict(self, X):
        return self.model.predict(self.pca.transform(X))


def tune_neighbor_count(X_bank, y_bank, X_val, y_val):
    def validation_r2(neighbor_count):
        model = KNNRegressor(neighbor_count).fit(X_bank, y_bank)
        return mean_r2(y_val, model.predict(X_val))

    return max(NEIGHBORS, key=validation_r2)


### 6.1 Training

Tune neighbor counts on the validation sessions and build the stimulation-off bank. `NEIGHBORS` is chosen by mean validation R².


In [ ]:
KNN_TUNE_K = 10  # calibration trials used to tune KNN


def tune_knn_neighbor_counts(variant, lag_count):
    val_sessions = split_sessions('val', 1)
    X_bank, y_bank = collect_windows(OFF_TRAIN_SESSIONS, 'train', variant, lag_count)
    X_val, y_val = collect_windows(val_sessions, 'query', variant, lag_count)

    calibration = SPLITS['calibration']
    cal = [load_calibration_windows(s, calibration[s]['subsets'][str(KNN_TUNE_K)][0],
                                    calibration[s]['pool'], variant, lag_count)
           for s in val_sessions]
    X_cal, y_cal = [np.concatenate(part) for part in zip(*cal)]

    return (tune_neighbor_count(X_bank, y_bank, X_val, y_val),
            tune_neighbor_count(X_cal, y_cal, X_val, y_val))


def train_knn(variant, lag_count):
    bank_neighbors, calibration_neighbors = tune_knn_neighbor_counts(variant, lag_count)
    bank_decoder = None
    if variant == RAW_VARIANT:
        X_bank, y_bank = collect_windows(OFF_TRAIN_SESSIONS, 'train', variant, lag_count)
        bank_decoder = KNNRegressor(bank_neighbors).fit(X_bank, y_bank)
    return bank_decoder, calibration_neighbors


### 6.3 Evaluation

Score the stimulation-off bank and the k-shot calibration model on the query set.

In [ ]:
def evaluate_knn(draws, variant, feature_tag, lag_count, sessions, bank_decoder, calibration_neighbors):
    rows = []
    for session in sessions:
        query, off = load_query_and_off_windows(session, variant, lag_count)
        if bank_decoder is not None:
            rows.append(score_decoder(bank_decoder, session, 'knn_off', feature_tag, query, off))
        for k, draw, cal in calibration_draws(draws, session, variant, lag_count):
            decoder = KNNRegressor(calibration_neighbors).fit(*cal)
            rows.append(score_decoder(decoder, session, 'knn_incontext', feature_tag,
                                      query, off, k, draw))
    return rows


## 7. MLP

Two hidden layers on the same cached features. It tests nonlinear capacity without temporal recurrence or adaptation.

### 7.1 Model

In [ ]:
MLP_DROPOUT = 0.3


class MLPRegressor(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 32), nn.GELU(), nn.Dropout(MLP_DROPOUT),
            nn.Linear(32, 16), nn.GELU(), nn.Dropout(MLP_DROPOUT),
            nn.Linear(16, 2),
        )

    def forward(self, windows):
        return self.layers(windows)

### 7.2 Training

Train on stimulation-off bins with early stopping on held-out trials.

In [ ]:
def train_mlp(train, val, seed):
    return train_window(MLPRegressor(train[0].shape[1]), train, val, seed)


### 7.3 Evaluation

Score each seed on the ON query set. The model is frozen; it never sees the k calibration trials.

In [ ]:
def evaluate_mlp(model, seed, test, feature_tag):
    rows = []
    for session, (query, off) in test.items():
        X_query, y_query = query
        X_off, y_off = off
        rows.append(score_row(session, 'mlp', feature_tag,
                              y_query, predict(model, X_query),
                              y_off, predict(model, X_off), seed=seed))
    return rows


def run_mlp(variant, feature_tag, lag_count, sessions, seeds, training_log_dir):
    train = collect_windows(OFF_TRAIN_SESSIONS, 'train', variant, lag_count)
    val = collect_windows(OFF_TRAIN_SESSIONS, 'val', variant, lag_count)
    test = {s: load_query_and_off_windows(s, variant, lag_count) for s in sessions}
    rows = []
    for seed in seeds:
        print('mlp seed', seed)
        model, summary = train_mlp(train, val, seed)
        save_checkpoint(model, f'mlp_seed{seed}')
        save_training_log(summary, 'mlp', seed, training_log_dir)
        rows += evaluate_mlp(model, seed, test, feature_tag)
    return rows


## 8. GRU (RNN)

A 128-unit GRU reads each 40-bin trial as a sequence. It tests temporal context without adaptation.

### 8.1 Model

In [ ]:
HIDDEN_SIZE = 128


class GRURegressor(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.gru = nn.GRU(input_size, HIDDEN_SIZE, batch_first=True)
        self.output = nn.Linear(HIDDEN_SIZE, 2)

    def forward(self, trials):
        hidden_states, _ = self.gru(trials)
        return self.output(hidden_states)

### 8.2 Training

Train on stimulation-off trials with per-bin masked MSE and early stopping.

In [ ]:
def gru_datasets(variant, sessions):
    def load_part(s, part):
        return load_split_dataset(s, part, variant, 0)

    train = concat_off_train(load_part, 'train')
    val = concat_off_train(load_part, 'val')
    test = {s: (load_part(s, 'query'), load_off_reference_dataset(s, variant, 0))
            for s in sessions}
    return train, val, test


def train_gru(train, val, seed):
    return train_trial(GRURegressor(train['features'].shape[-1]), train, val, seed)


### 8.3 Evaluation

Score each seed on the ON query set. The GRU is frozen and does not use the k calibration trials.

In [ ]:
def evaluate_gru(model, seed, test, feature_tag):
    return [score_trial(model, s, 'gru', feature_tag, seed, *data)
            for s, data in test.items()]


def run_gru(variant, feature_tag, sessions, seeds, training_log_dir):
    train, val, test = gru_datasets(variant, sessions)
    rows = []
    for seed in seeds:
        print('gru seed', seed)
        model, summary = train_gru(train, val, seed)
        save_checkpoint(model, f'gru_seed{seed}')
        save_training_log(summary, 'gru', seed, training_log_dir)
        rows += evaluate_gru(model, seed, test, feature_tag)
    return rows


## 9. EEGNet

A causal EEGNet on raw 1 kHz LFP. It learns temporal and spatial filters instead of using hand-designed bands.

### 9.1 Model

In [ ]:
TEMPORAL_KERNEL_SAMPLES = 500
SEPARABLE_KERNEL_SAMPLES = 62
TEMPORAL_FILTERS = 8
SPATIAL_FILTERS = 16
SEPARABLE_FILTERS = 16
FIRST_POOL_SAMPLES = 2
SECOND_POOL_SAMPLES = 25
EEGNET_DROPOUT = 0.25


def _left_only_padding(kernel_samples):
    return nn.ZeroPad2d((kernel_samples - 1, 0, 0, 0))


class EEGNetRegressor(nn.Module):
    def __init__(self, channel_count):
        super().__init__()
        self.layers = nn.Sequential(
            _left_only_padding(TEMPORAL_KERNEL_SAMPLES),
            nn.Conv2d(1, TEMPORAL_FILTERS, (1, TEMPORAL_KERNEL_SAMPLES), bias=False),
            nn.BatchNorm2d(TEMPORAL_FILTERS),
            nn.Conv2d(TEMPORAL_FILTERS, SPATIAL_FILTERS, (channel_count, 1),
                      groups=TEMPORAL_FILTERS, bias=False),
            nn.BatchNorm2d(SPATIAL_FILTERS),
            nn.ELU(),
            nn.AvgPool2d((1, FIRST_POOL_SAMPLES)),
            nn.Dropout(EEGNET_DROPOUT),
            _left_only_padding(SEPARABLE_KERNEL_SAMPLES),
            nn.Conv2d(SPATIAL_FILTERS, SPATIAL_FILTERS, (1, SEPARABLE_KERNEL_SAMPLES),
                      groups=SPATIAL_FILTERS, bias=False),
            nn.Conv2d(SPATIAL_FILTERS, SEPARABLE_FILTERS, (1, 1), bias=False),
            nn.BatchNorm2d(SEPARABLE_FILTERS),
            nn.ELU(),
            nn.AvgPool2d((1, SECOND_POOL_SAMPLES)),
            nn.Dropout(EEGNET_DROPOUT),
            nn.Conv2d(SEPARABLE_FILTERS, 2, (1, 1)),
        )

    def forward(self, lfp):
        return self.layers(lfp.unsqueeze(1)).squeeze(2).transpose(1, 2)

### 9.2 Training

Load 2 s raw-LFP trials, then train with early stopping on held-out stimulation-off trials. `Butters-250711-101424` has no screen-cm kinematics, so it is excluded from validation.

In [ ]:
SESSION_WITHOUT_KINEMATICS = 'Butters-250711-101424'


def load_lfp_split(session, part):
    return load_lfp_dataset(session, load_split_trials(session, part))


def eegnet_datasets(sessions):
    train = concat_off_train(load_lfp_split, 'train')
    val = concat_off_train(load_lfp_split, 'val', SESSION_WITHOUT_KINEMATICS)
    off = SPLITS['off_reference']
    test = {s: (load_lfp_split(s, 'query'), load_lfp_dataset(off[s], None)) for s in sessions}
    return train, val, test


def train_eegnet(train, val, seed, epochs):
    return train_trial(EEGNetRegressor(train['features'].shape[1]), train, val, seed, epochs)


### 9.3 Evaluation

Score each seed on the ON query set. The network is frozen and does not use the k calibration trials.

In [ ]:
def evaluate_eegnet(model, seed, test, feature_tag='raw'):
    return [score_trial(model, s, 'eegnet', feature_tag, seed, *data)
            for s, data in test.items()]


def run_eegnet(sessions, seeds, epochs, training_log_dir):
    train, val, test = eegnet_datasets(sessions)
    rows = []
    for seed in seeds:
        print('eegnet seed', seed)
        model, summary = train_eegnet(train, val, seed, epochs)
        save_checkpoint(model, f'eegnet_seed{seed}')
        save_training_log(summary, 'eegnet', seed, training_log_dir)
        rows += evaluate_eegnet(model, seed, test)
    return rows


## 10. Run all baselines

Dispatch to the per-predictor sections above and write one parquet file.

In [ ]:
def run_eval(draws_path=DRAWS_PATH,
             out=os.path.join(RESULTS_DIR, 'results.parquet'),
             methods=('ridge',), n_lags=9, sessions=None, seed_count=5, epochs=150,
             training_log_dir=None):
    draws = read_json(draws_path)
    sessions = sessions or split_sessions('test', 1)
    seeds = range(seed_count)
    rows = []

    if 'ridge' in methods:
        print('ridge')
        for variant, feature_tag in VARIANTS:
            rows += run_ridge(draws, variant, feature_tag, n_lags, sessions)
    if 'mlp' in methods:
        rows += run_mlp(RAW_VARIANT, 'raw', n_lags, sessions, seeds, training_log_dir)
    if 'gru' in methods:
        rows += run_gru(RAW_VARIANT, 'raw', sessions, seeds, training_log_dir)
    if 'eegnet' in methods:
        rows += run_eegnet(sessions, seeds, epochs, training_log_dir)
    if 'knn' in methods:
        print('knn')
        bank, calibration_neighbors = train_knn(RAW_VARIANT, n_lags)
        for variant, feature_tag in VARIANTS:
            rows += evaluate_knn(draws, variant, feature_tag, n_lags, sessions,
                                 bank if variant == RAW_VARIANT else None, calibration_neighbors)

    os.makedirs(os.path.dirname(out), exist_ok=True)
    pd.DataFrame(rows).to_parquet(out, index=False)


## 11. Predictions

In [ ]:
LAG_COUNT = 9
RIDGE_ALPHA = 1e5  # fixed penalty for prediction plots
KNN_NEIGHBORS = 100  # fixed KNN k for prediction plots
PRE_S = 0.5  # seconds before go cue
DECODER_COLORS = ['#7f7f7f', '#d62728', '#2ca02c']
DEFAULT_SESSIONS = 'Butters-250616-102124,Butters-250520-101021'


def measured_positions(session, bin_ids):
    kinematics = load_kinematics(session)
    flat_bin_ids = bin_ids.reshape(-1)
    x_positions = kinematics['Index_x_cm'].reindex(flat_bin_ids).to_numpy(np.float64)
    y_positions = kinematics['Index_y_cm'].reindex(flat_bin_ids).to_numpy(np.float64)
    return np.stack([x_positions, y_positions], axis=-1).reshape(*bin_ids.shape, 2)


def decode_session(session, draws_path):
    draws = read_json(draws_path)
    entry = draws['sessions'][session]
    largest_calibration_size = max(draws['ks'])

    train = collect_windows(OFF_TRAIN_SESSIONS, 'train', RAW_VARIANT, LAG_COUNT)
    query = load_split_dataset(session, 'query', RAW_VARIANT, LAG_COUNT)
    query_features, _ = valid_windows(query)
    calibration_windows = load_calibration_windows(
        session, entry['subsets'][str(largest_calibration_size)][0], entry['pool'], RAW_VARIANT, LAG_COUNT)

    decoders = {
        'ridge, no adaptation': RidgeRegressor(RIDGE_ALPHA).fit(*train),
        f'ridge, refit on k={largest_calibration_size}':
            RidgeRegressor(RIDGE_ALPHA).fit(*calibration_windows),
        f'KNN, in-context k={largest_calibration_size}':
            KNNRegressor(KNN_NEIGHBORS).fit(*calibration_windows),
    }

    measured = query['targets'].astype(np.float64)
    predictions = {}
    for name, decoder in decoders.items():
        predicted_trials = np.full(measured.shape, np.nan)
        predicted_trials[query['valid_bins']] = decoder.predict(query_features)
        predictions[name] = predicted_trials
    return dict(session=session, measured=measured, predictions=predictions,
                valid_bins=query['valid_bins'], trials=query['trials'],
                positions=measured_positions(session, query['bin_ids']))


def select_clearest_trials(result, count):
    valid_bin_counts = result['valid_bins'].sum(axis=1)
    mean_speed = np.nanmean(speed(result['measured']), axis=1)
    return np.lexsort((-np.nan_to_num(mean_speed), -valid_bin_counts))[:count]


def decoder_r2(result, name):
    valid_bins = result['valid_bins']
    return mean_r2(result['measured'][valid_bins], result['predictions'][name][valid_bins])


def speed(velocity):
    return np.linalg.norm(velocity, axis=-1)


def bin_times(result):
    return np.arange(result['measured'].shape[1]) * BIN_S - PRE_S


def plot_measured_and_decoded(axis, times, measured_trace, decoded_traces):
    axis.plot(times, measured_trace, color='k', linewidth=2.4, label='measured')
    for color, (name, trace) in zip(DECODER_COLORS, decoded_traces.items()):
        axis.plot(times, trace, color=color, linewidth=1.3, label=name)
    axis.set_xlim(times[0], times[-1])
    axis.grid(alpha=0.25)


def plot_session_summary(result, out_path):
    names = list(result['predictions'])
    times = bin_times(result)
    trial_index = select_clearest_trials(result, 1)[0]
    trial_number = result['trials'][trial_index]
    measured_trial = result['measured'][trial_index]
    decoded_trial = {name: result['predictions'][name][trial_index] for name in names}

    figure = plt.figure(figsize=(13, 7.4))
    grid = figure.add_gridspec(2, 3, hspace=0.42, wspace=0.32)

    for column, component in enumerate(('vx', 'vy')):
        axis = figure.add_subplot(grid[0, column])
        plot_measured_and_decoded(axis, times, measured_trial[:, column],
                                  {name: trace[:, column] for name, trace in decoded_trial.items()})
        axis.axhline(0, color='k', linewidth=0.5, alpha=0.4)
        axis.set_xlabel('time from go cue (s)')
        axis.set_ylabel(f'Index {component} (cm/s)')
        axis.set_title(f'Trial {trial_number} — velocity {component}', fontsize=9)
        if column == 0:
            axis.legend(fontsize=7)

    axis = figure.add_subplot(grid[0, 2])
    plot_measured_and_decoded(axis, times, speed(measured_trial),
                              {name: speed(trace) for name, trace in decoded_trial.items()})
    axis.set_xlabel('time from go cue (s)')
    axis.set_ylabel('speed (cm/s)')
    axis.set_title(f'Trial {trial_number} — speed', fontsize=9)
    axis.legend(fontsize=7)

    axis = figure.add_subplot(grid[1, 0])
    measured_bins = result['measured'][result['valid_bins']]
    for color, name in zip(DECODER_COLORS, names):
        decoded_bins = result['predictions'][name][result['valid_bins']]
        axis.scatter(measured_bins[:, 0], decoded_bins[:, 0], s=4, alpha=0.25, color=color,
                     label=f'{name} (R²={decoder_r2(result, name):+.3f})')
    limit = np.nanpercentile(np.abs(measured_bins[:, 0]), 99)
    axis.plot([-limit, limit], [-limit, limit], color='k', linewidth=0.8)
    axis.set_xlim(-limit, limit)
    axis.set_ylim(-limit, limit)
    axis.set_xlabel('measured vx (cm/s)')
    axis.set_ylabel('decoded vx (cm/s)')
    axis.set_title('All query bins', fontsize=9)
    axis.legend(fontsize=6, loc='upper left')
    axis.grid(alpha=0.25)

    axis = figure.add_subplot(grid[1, 1])
    plot_measured_and_decoded(axis, times, np.nanmean(speed(result['measured']), axis=0),
                              {name: np.nanmean(speed(result['predictions'][name]), axis=0)
                               for name in names})
    axis.set_xlabel('time from go cue (s)')
    axis.set_ylabel('mean speed (cm/s)')
    axis.set_title(f'Mean speed profile over {len(result["trials"])} query trials', fontsize=9)
    axis.legend(fontsize=7)

    axis = figure.add_subplot(grid[1, 2])
    r2_values = [decoder_r2(result, name) for name in names]
    axis.barh(range(len(names)), r2_values, color=DECODER_COLORS, height=0.6)
    for row, (name, value) in enumerate(zip(names, r2_values)):
        label = name.replace('ridge, ', '').replace('KNN, in-context', 'KNN')
        axis.text(value + 0.002, row, f'{label}  {value:+.3f}', va='center', ha='left', fontsize=7)
    axis.set_yticks([])
    axis.set_xlim(0, max(r2_values) * 1.9)
    axis.axvline(0, color='k', linewidth=0.8)
    axis.set_xlabel('R² (2-target mean, query bins)')
    axis.set_title('Decoder accuracy on this session', fontsize=9)
    axis.grid(alpha=0.25, axis='x')

    figure.suptitle(f'{result["session"]} — decoded Index-finger action', fontsize=11)
    figure.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(figure)


def plot_trial_speeds(result, out_path):
    times = bin_times(result)
    figure, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
    for axis, trial_index in zip(axes.ravel(), select_clearest_trials(result, 4)):
        plot_measured_and_decoded(axis, times, speed(result['measured'][trial_index]),
                                  {name: speed(trace[trial_index])
                                   for name, trace in result['predictions'].items()})
        axis.set_title(f'Trial {result["trials"][trial_index]}', fontsize=9)
        axis.set_ylabel('speed (cm/s)')
    for axis in axes[-1]:
        axis.set_xlabel('time from go cue (s)')
    axes[0, 0].legend(fontsize=7)
    figure.suptitle(f'{result["session"]} — speed envelope, measured vs decoded', fontsize=11)
    figure.tight_layout()
    figure.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(figure)


def run_predictions(draws_path=DRAWS_PATH,
                    sessions=DEFAULT_SESSIONS,
                    out_dir=os.path.join(RESULTS_DIR, 'predictions')):
    os.makedirs(out_dir, exist_ok=True)
    for session in sessions.split(','):
        result = decode_session(session, draws_path)
        plot_session_summary(result, os.path.join(out_dir, f'{session}_actions.png'))
        plot_trial_speeds(result, os.path.join(out_dir, f'{session}_trials_speed.png'))
        np.savez_compressed(os.path.join(out_dir, f'{session}_predictions.npz'),
                            measured=result['measured'], mask=result['valid_bins'],
                            trials=result['trials'], positions=result['positions'],
                            **{f'pred__{name}': trace for name, trace in result['predictions'].items()})

## 12. Figures

In [ ]:
METHOD_LABELS = {
    'ridge_raw_off': 'Ridge, raw, no adapt',
    'ridge_parrm_off': 'Ridge, PARRM, no adapt',
    'ridge_raw_refit': 'Ridge, raw, refit',
    'ridge_parrm_refit': 'Ridge, PARRM, refit',
    'knn_off': 'KNN, stim-off bank',
    'knn_incontext': 'KNN, k calibration',
    'mlp': 'MLP',
    'eegnet': 'EEGNet',
    'gru': 'GRU',
}
SHORT_METHOD_LABELS = {
    'ridge_raw_off': 'Ridge raw\nno adapt',
    'ridge_parrm_off': 'Ridge PARRM\nno adapt',
    'ridge_raw_refit': 'Ridge raw\nrefit k=20',
    'ridge_parrm_refit': 'Ridge PARRM\nrefit k=20',
    'knn_off': 'KNN\nstim-off bank',
    'knn_incontext': 'KNN\nk calib.',
    'mlp': 'MLP',
    'eegnet': 'EEGNet',
    'gru': 'GRU',
}
METHOD_COLORS = {
    'ridge_raw_off': '#4c72b0', 'ridge_parrm_off': '#8ab4d8',
    'ridge_raw_refit': '#c44e52', 'ridge_parrm_refit': '#e08a7f',
    'knn_off': '#8172b2', 'knn_incontext': '#55a868',
    'mlp': '#937860', 'eegnet': '#ccb974', 'gru': '#6b6b6b',
}
NON_ADAPTIVE_METHODS = ['ridge_raw_off', 'ridge_parrm_off', 'knn_off', 'mlp', 'eegnet', 'gru']
ADAPTIVE_METHODS = ['ridge_raw_refit', 'ridge_parrm_refit', 'knn_incontext']
METHOD_ORDER = NON_ADAPTIVE_METHODS + ['knn_incontext', 'ridge_raw_refit', 'ridge_parrm_refit']
RESULT_KEYS = ['method', 'feature_variant', 'session', 'k', 'draw', 'seed']
RIDGE_FACTORIAL_CELLS = [('raw', 'no refit', 'ridge_raw_off', 0),
                         ('raw', 'refit k=20', 'ridge_raw_refit', 20),
                         ('PARRM', 'no refit', 'ridge_parrm_off', 0),
                         ('PARRM', 'refit k=20', 'ridge_parrm_refit', 20)]
MAIN_TABLE_COLUMNS = {'method_label': 'method',
                      'r2_mean_3Hz': 'R² 3 Hz', 'r2_mean_20Hz': 'R² 20 Hz',
                      'r2_off_mean_3Hz': 'off 3 Hz', 'r2_off_mean_20Hz': 'off 20 Hz'}
K_MAX = 20  # largest calibration set


def load_results(pattern):
    frames = [pd.read_parquet(path) for path in sorted(glob.glob(pattern))]
    return pd.concat(frames, ignore_index=True).drop_duplicates(subset=RESULT_KEYS)


def average_per_session(results):
    return (results.groupby(['method', 'am_freq', 'k', 'session'], dropna=False)
            .agg(r2=('r2_ave', 'mean'), r2_off=('r2_off_ave', 'mean'))
            .reset_index())


def bootstrap_means(values, resample_count):
    return np.random.default_rng(0).choice(values, size=(resample_count, len(values)),
                                           replace=True).mean(axis=1)


def bootstrap_interval(values, resample_count=2000):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan, np.nan
    resampled = bootstrap_means(values, resample_count)
    return float(np.percentile(resampled, 2.5)), float(np.percentile(resampled, 97.5))


def select_sessions(session_scores, method, frequency, k):
    return session_scores[(session_scores.method == method) &
                          (session_scores.am_freq == frequency) &
                          (session_scores.k == k)]


def session_r2(session_scores, method, frequency, k):
    return select_sessions(session_scores, method, frequency, k).r2.to_numpy()


def stimulation_frequencies(session_scores):
    return sorted(session_scores.am_freq.unique())


def save_figure(figure, path):
    figure.tight_layout()
    figure.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(figure)


def plot_r2_vs_calibration_trials(session_scores, path):
    figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), sharey=True)
    for axis, frequency in zip(axes, stimulation_frequencies(session_scores)):
        for method in NON_ADAPTIVE_METHODS:
            mean_r2 = np.nanmean(session_r2(session_scores, method, frequency, 0))
            axis.axhline(mean_r2, linestyle='--', linewidth=1.4, color=METHOD_COLORS[method],
                         label=f'{METHOD_LABELS[method]}  {mean_r2:+.3f}')
        for method in ADAPTIVE_METHODS:
            calibration_sizes = sorted(session_scores[(session_scores.method == method) &
                                                      (session_scores.am_freq == frequency)].k.unique())
            values_by_size = [session_r2(session_scores, method, frequency, k)
                              for k in calibration_sizes]
            lows, highs = zip(*[bootstrap_interval(values) for values in values_by_size])
            axis.plot(calibration_sizes, [np.nanmean(values) for values in values_by_size],
                      marker='o', color=METHOD_COLORS[method], label=METHOD_LABELS[method])
            axis.fill_between(calibration_sizes, lows, highs, color=METHOD_COLORS[method], alpha=0.16)
        axis.set_title(f'{int(frequency)} Hz AM')
        axis.set_xlabel('calibration trials k')
        axis.set_xticks([0, 2, 5, 10, 20])
        axis.grid(alpha=0.25)
        axis.set_xlim(0, 21.6)
        axis.set_ylim(-0.05, 0.09)
    axes[0].set_ylabel('R² (Index velocity, 2-target mean)')
    axes[0].legend(fontsize=6.2, loc='lower right', framealpha=0.9)
    figure.suptitle('Decoding R² vs number of labelled stimulation trials — dashed = no adaptation; '
                    'bands = 95% session bootstrap',
                    fontsize=8.6)
    save_figure(figure, path)


def draw_method_bar(axis, position, width, values, **style):
    mean_r2 = np.nanmean(values) if len(values) else np.nan
    low, high = bootstrap_interval(values)
    axis.bar(position, mean_r2, width, edgecolor='k', linewidth=0.5,
             yerr=[[mean_r2 - low], [high - mean_r2]], capsize=2, error_kw=dict(lw=0.7), **style)


def plot_method_comparison(session_scores, path):
    paired_bar_width = 0.36
    figure, axes = plt.subplots(1, 2, figsize=(12.4, 4.6), sharey=True)
    for axis, frequency in zip(axes, stimulation_frequencies(session_scores)):
        for position, method in enumerate(METHOD_ORDER):
            color = METHOD_COLORS[method]
            if method in NON_ADAPTIVE_METHODS:
                draw_method_bar(axis, position, 0.72, session_r2(session_scores, method, frequency, 0),
                                color=color, hatch='///')
                continue
            for offset, k in zip([-paired_bar_width, paired_bar_width], [0, K_MAX]):
                draw_method_bar(axis, position + offset, paired_bar_width,
                                session_r2(session_scores, method, frequency, k), color=color,
                                alpha=1.0 if k == K_MAX else 0.45)
        axis.set_xticks(np.arange(len(METHOD_ORDER)))
        axis.set_xticklabels([SHORT_METHOD_LABELS[method] for method in METHOD_ORDER], fontsize=6.4)
        axis.axhline(0, color='k', linewidth=0.8)
        axis.set_title(f'{int(frequency)} Hz AM')
        axis.grid(alpha=0.25, axis='y')
    axes[0].set_ylabel('R² (2-target mean)')
    axes[0].legend(handles=[Patch(facecolor='w', edgecolor='k', hatch='///', label='no adaptation'),
                            Patch(facecolor='grey', edgecolor='k', label='k = 20'),
                            Patch(facecolor='grey', edgecolor='k', alpha=0.45, label='k = 0')],
                   fontsize=7, loc='lower left')
    figure.suptitle('Baseline comparison on held-out stimulation sessions '
                    '(error bars: 95% session bootstrap)', fontsize=9.5)
    save_figure(figure, path)


def plot_ridge_factorial(session_scores, path):
    figure, axes = plt.subplots(1, 2, figsize=(9.2, 3.9))
    for axis, frequency in zip(axes, stimulation_frequencies(session_scores)):
        mean_r2 = np.array([[np.nanmean(session_r2(session_scores, method, frequency, k))
                             for _, _, method, k in RIDGE_FACTORIAL_CELLS[start:start + 2]]
                            for start in (0, 2)])
        image = axis.imshow(mean_r2, cmap='RdYlGn', vmin=-0.03, vmax=0.06, aspect='auto')
        for (row, column), value in np.ndenumerate(mean_r2):
            axis.text(column, row, f'{value:+.3f}', ha='center', va='center', fontsize=11,
                      fontweight='bold', color='k')
        axis.set_xticks([0, 1])
        axis.set_xticklabels(['decoder from\nother days', 'refit on\nk=20 ON'])
        axis.set_yticks([0, 1])
        axis.set_yticklabels(['raw', 'PARRM\ncleaned'])
        axis.set_title(f'{int(frequency)} Hz AM')
        axis.set_xlabel('adaptation')
        axis.set_ylabel('input cleaning')
        figure.colorbar(image, ax=axis, fraction=0.046, pad=0.03).set_label('R²', fontsize=8)
    figure.suptitle('Ridge factorial: does cleaning help, or do labels?', fontsize=10)
    save_figure(figure, path)


def plot_session_heatmap(session_scores, path):
    method_rows = []
    for method in METHOD_ORDER:
        k = 0 if method in NON_ADAPTIVE_METHODS else K_MAX
        selected = session_scores[(session_scores.method == method) & (session_scores.k == k)]
        method_rows.append(selected.set_index('session').r2)
    session_order = (session_scores[session_scores.method == 'ridge_raw_off']
                     .sort_values('r2').session.tolist())
    matrix = pd.concat(method_rows, axis=1).T[session_order]

    figure, axis = plt.subplots(figsize=(12, 3.6))
    image = axis.imshow(matrix.to_numpy(), cmap='RdBu_r', aspect='auto',
                        norm=TwoSlopeNorm(vmin=-0.16, vcenter=0.0, vmax=0.16))
    axis.set_yticks(range(len(METHOD_ORDER)))
    axis.set_yticklabels([METHOD_LABELS[method] for method in METHOD_ORDER], fontsize=7.5)
    axis.set_xticks(range(len(session_order)))
    axis.set_xticklabels([session.replace('Butters-', '') for session in session_order],
                         rotation=90, fontsize=7)
    axis.set_title('Per-session R² (k=20 when the method adapts); sessions sorted by '
                   'no-adaptation ridge', fontsize=9)
    figure.colorbar(image, ax=axis, fraction=0.022, pad=0.012).set_label('R²', fontsize=8)
    save_figure(figure, path)


def plot_training_curves(training_log_dir, path):
    training_log_files = sorted(glob.glob(os.path.join(training_log_dir, '*.json')))
    figure, axis = plt.subplots(figsize=(7.6, 4.2))
    for training_log_file in training_log_files:
        record = read_json(training_log_file)
        color = METHOD_COLORS[record['method']]
        axis.plot([epoch['epoch'] for epoch in record['training_log']],
                  [epoch['val_r2'] for epoch in record['training_log']],
                  linewidth=1.1, color=color, alpha=0.85)
        axis.scatter([record['best_epoch']], [record['best_val_r2']], color=color, s=18, zorder=5,
                     edgecolor='k', linewidth=0.4)
    axis.legend(handles=[plt.Line2D([], [], color=METHOD_COLORS[method], label=method)
                         for method in ('mlp', 'gru', 'eegnet')
                         if any(method in os.path.basename(name) for name in training_log_files)],
                fontsize=8)
    axis.set_xlabel('epoch')
    axis.set_ylabel('validation R² (held-out block of training sessions)')
    axis.set_title('Training curves; dots mark the early-stopping checkpoint. '
                   'High validation R² does not transfer across days.', fontsize=8.5)
    axis.grid(alpha=0.25)
    save_figure(figure, path)


def plot_noise_floor(results, results_dir, path):
    stimulated_sessions_by_off_session = {}
    for stimulated_session, off_session in SPLITS['off_reference'].items():
        stimulated_sessions_by_off_session.setdefault(off_session, []).append(stimulated_session)
    cross_day_r2 = results[results.method == 'ridge_raw_off'].groupby('session').r2_off_ave.mean()
    diagnostic = pd.read_csv(os.path.join(results_dir, 'diagnostic_within_day_off.csv'))
    diagnostic['cross_day'] = diagnostic.session.map(
        lambda off_session: cross_day_r2.reindex(
            stimulated_sessions_by_off_session.get(off_session, [])).mean())

    figure, axis = plt.subplots(figsize=(7.4, 3.8))
    positions = np.arange(len(diagnostic))
    axis.bar(positions - 0.2, diagnostic.within_day_r2, 0.4, color='#4c72b0',
             label='same-day fit, held-out trials')
    axis.bar(positions + 0.2, diagnostic.cross_day, 0.4, color='#c44e52',
             label='decoder from other days')
    axis.set_xticks(positions)
    axis.set_xticklabels([session.replace('Butters-', '') for session in diagnostic.session],
                         rotation=20, fontsize=7.5)
    axis.axhline(0, color='k', linewidth=0.8)
    axis.set_ylabel('R² (no stimulation)')
    axis.set_title('The noise floor: even without stimulation, accuracy is ~0.07', fontsize=9.5)
    axis.legend(fontsize=8)
    axis.grid(alpha=0.25, axis='y')
    save_figure(figure, path)


def build_recovery_table(results, resample_count=500):
    reference = (results[results.method == 'ridge_raw_off'][['session', 'r2_ave', 'r2_off_ave']]
                 .drop_duplicates('session').set_index('session'))
    rows = []
    for (method, frequency, k), group in results.groupby(['method', 'am_freq', 'k']):
        fractions = np.array([
            float(recovery(session_rows.r2_ave.mean(),
                           reference.loc[session].r2_ave,
                           reference.loc[session].r2_off_ave))
            for session, session_rows in group.groupby('session')])
        resampled = bootstrap_means(fractions, resample_count)
        low, high = np.nanpercentile(resampled, [2.5, 97.5])
        rows.append(dict(method=method, am_freq=frequency, k=int(k), n_sessions=len(fractions),
                         rho=float(np.nanmean(fractions)), rho_lo=float(low), rho_hi=float(high)))
    return pd.DataFrame(rows)


def plot_recovery(table, path):
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    adaptive = table[table.method.isin(ADAPTIVE_METHODS)]
    for axis, (frequency, frequency_rows) in zip(axes, adaptive.groupby('am_freq')):
        for method, series in frequency_rows.groupby('method'):
            series = series.sort_values('k')
            axis.errorbar(series.k, series.rho,
                          yerr=[series.rho - series.rho_lo, series.rho_hi - series.rho],
                          marker='o', capsize=3, color=METHOD_COLORS[method],
                          label=METHOD_LABELS[method])
        axis.axhline(0, color='k', linewidth=0.8)
        axis.axhline(1, color='k', linewidth=0.8, linestyle=':')
        axis.set_title(f'{int(frequency)} Hz AM')
        axis.set_xlabel('calibration trials k')
        axis.set_ylabel('fraction of loss recovered (rho)')
        axis.grid(alpha=0.3)
        axis.set_xticks([2, 5, 10, 20])
    axes[0].legend(fontsize=7)
    save_figure(figure, path)


def write_main_tables(session_scores, out_dir):
    rows = []
    for method in METHOD_ORDER:
        calibration_sizes = ([0] if method in NON_ADAPTIVE_METHODS else
                             sorted(session_scores[session_scores.method == method].k.unique()))
        for k in calibration_sizes:
            for frequency in stimulation_frequencies(session_scores):
                selected = select_sessions(session_scores, method, frequency, k)
                values = selected.r2.to_numpy()
                low, high = bootstrap_interval(values)
                rows.append(dict(method=method, method_label=METHOD_LABELS[method],
                                 am_freq_hz=frequency, k=int(k), n_sessions=len(values),
                                 r2_mean=np.nanmean(values), r2_ci_lo=low, r2_ci_hi=high,
                                 r2_off_mean=np.nanmean(selected.r2_off),
                                 r2_sem=np.nanstd(values, ddof=1) / np.sqrt(len(values))))
    table = pd.DataFrame(rows)
    table.to_csv(os.path.join(out_dir, 'table_main.csv'), index=False)

    wide = table.pivot_table(index=['method_label', 'k'], columns='am_freq_hz',
                             values=['r2_mean', 'r2_off_mean'])
    wide.columns = [f'{metric}_{int(frequency)}Hz' for metric, frequency in wide.columns]
    wide = wide.reset_index()
    label_rank = {label: rank for rank, label in enumerate(METHOD_LABELS.values())}
    wide = (wide.assign(label_rank=wide.method_label.map(label_rank))
            .sort_values(['label_rank', 'k'])
            .drop(columns='label_rank')
            .rename(columns=MAIN_TABLE_COLUMNS)
            .round(4))
    with open(os.path.join(out_dir, 'table_main.md'), 'w') as file:
        file.write('# Main results\n\n')
        file.write('R² on the fixed 30-trial query set; mean over sessions, draws and seeds. '
                   '`off` is the same decoder on that day\'s no-stimulation session.\n\n')
        file.write(wide.to_markdown(index=False))
        file.write('\n')
    with open(os.path.join(out_dir, 'table_main.tex'), 'w') as file:
        file.write(wide.to_latex(index=False, float_format='%.4f', escape=True))


def write_ridge_factorial_table(session_scores, out_dir):
    rows = []
    for cleaning, adaptation, method, k in RIDGE_FACTORIAL_CELLS:
        row = dict(cleaning=cleaning, adaptation=adaptation, method=method, k=k)
        for frequency in stimulation_frequencies(session_scores):
            row[f'r2_{int(frequency)}Hz'] = np.nanmean(session_r2(session_scores, method, frequency, k))
        rows.append(row)
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, 'table_ridge_factorial.csv'), index=False)


def run_figures(pattern=os.path.join(RESULTS_DIR, 'results*.parquet'),
                out_dir=RESULTS_DIR,
                training_log_dir=os.path.join(RESULTS_DIR, 'training_logs')):
    def output_path(filename):
        return os.path.join(out_dir, filename)

    results = load_results(pattern)
    session_scores = average_per_session(results)
    recovery_table = build_recovery_table(results)

    plot_r2_vs_calibration_trials(session_scores, output_path('fig1_r2_vs_k.png'))
    plot_method_comparison(session_scores, output_path('fig2_method_comparison.png'))
    plot_ridge_factorial(session_scores, output_path('fig3_ridge_factorial.png'))
    plot_session_heatmap(session_scores, output_path('fig4_session_heatmap.png'))
    plot_training_curves(training_log_dir, output_path('fig5_training_curves.png'))
    plot_noise_floor(results, out_dir, output_path('fig6_noise_floor.png'))
    plot_recovery(recovery_table, output_path('recovery_vs_k.png'))

    recovery_table.to_csv(output_path('table_recovery.csv'), index=False)
    write_main_tables(session_scores, out_dir)
    write_ridge_factorial_table(session_scores, out_dir)
    session_scores.pivot_table(index='session', columns=['method', 'k'], values='r2').to_csv(
        output_path('table_per_session.csv'))

## 13. Full pipeline

In [ ]:
RUN_PIPELINE = False  # True retrains every model and overwrites files in results/

if RUN_PIPELINE:
    training_log_dir = os.path.join(RESULTS_DIR, 'training_logs')
    run_draws()
    run_eval(methods=['ridge', 'knn'], out=os.path.join(RESULTS_DIR, 'results_ridge_knn.parquet'))
    for method in ('mlp', 'gru', 'eegnet'):  # eegnet takes about 1 hour
        run_eval(methods=[method], out=os.path.join(RESULTS_DIR, f'results_{method}.parquet'),
                 training_log_dir=training_log_dir)
    run_figures()
    run_predictions()